In [0]:
print("Ambiente pronto para a Orbital!")

Ambiente pronto para a Orbital!


In [0]:
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
from collections import defaultdict
import plotly.graph_objects as go
import plotly.express as px

# Ajuste de display do Pandas para facilitar a leitura no Databricks
pd.set_option('display.max_columns', None)

In [0]:
# Puxando a tabela bruta do BigQuery
nome_tabela = "workspace.default.bq_results_20260617_234748_1781740105311" 
df_ga4 = spark.table(nome_tabela)

# Marcando quem comprou (1) e quem não comprou (0) e limpando nulos
df_com_conversao = df_ga4.withColumn(
    "comprou",
    F.when(F.col("event_name") == "purchase", 1).otherwise(0)
).filter(F.col("utm_source").isNotNull())

In [0]:
# Agrupando tudo por usuário para montar o caminho que ele fez
df_jornada_final = df_com_conversao.groupBy("user_pseudo_id").agg(
    F.array_join(F.collect_list("utm_source"), " > ").alias("caminho"),
    F.max("comprou").alias("conversao")
)

# Agrupando por caminhos idênticos (reduz o tamanho do df pra levar pra memória local)
df_markov_ready = df_jornada_final.groupBy("caminho").agg(
    F.sum("conversao").alias("total_conversoes"),
    F.count("*").alias("total_jornadas")
)

In [0]:
# Trazendo pro Pandas pra facilitar a limpeza de string
df_pandas = df_markov_ready.toPandas()

def limpar_caminho(caminho):
    if not isinstance(caminho, str): return ""
    canais = [c.strip() for c in caminho.split('>')]
    
    canais_limpos = []
    for c in canais:
        # Removendo <Other> e self-transitions (cliques seguidos no mesmo canal)
        if c and c != '<Other>':
            if not canais_limpos or c != canais_limpos[-1]:
                canais_limpos.append(c)
                
    return ' > '.join(canais_limpos)

df_pandas['caminho_limpo'] = df_pandas['caminho'].apply(limpar_caminho)

# Consolida os números finais após a limpeza
df_modelagem = df_pandas.groupby('caminho_limpo')[['total_conversoes', 'total_jornadas']].sum().reset_index()
df_modelagem = df_modelagem[df_modelagem['caminho_limpo'] != '']

print(f"Jornadas únicas prontas para modelagem: {len(df_modelagem)}")

Jornadas únicas prontas para modelagem: 100


In [0]:
# Adicionando os nós de início e fim da jornada
caminhos_com_resultado = []
for _, row in df_modelagem.iterrows():
    caminho = row['caminho_limpo']
    conversoes = int(row['total_conversoes'])
    jornadas = int(row['total_jornadas'])
    
    caminhos_com_resultado.extend([f"Start > {caminho} > Conversion"] * conversoes)
    caminhos_com_resultado.extend([f"Start > {caminho} > Null"] * (jornadas - conversoes))

# Contagem de transições (A -> B)
transicoes = defaultdict(int)
total_saidas = defaultdict(int)
estados_unicos = set()

for caminho in caminhos_com_resultado:
    nos = caminho.split(' > ')
    for i in range(len(nos) - 1):
        estado_atual, proximo_estado = nos[i], nos[i+1]
        transicoes[(estado_atual, proximo_estado)] += 1
        total_saidas[estado_atual] += 1
        estados_unicos.update([estado_atual, proximo_estado])

# Montagem da matriz
estados_lista = sorted(list(estados_unicos))
matriz = pd.DataFrame(np.zeros((len(estados_lista), len(estados_lista))), 
                      index=estados_lista, columns=estados_lista)

for (estado_atual, proximo_estado), contagem in transicoes.items():
    matriz.at[estado_atual, proximo_estado] = contagem / total_saidas[estado_atual]

# Estados absorventes
for estado in ['Conversion', 'Null']:
    if estado in matriz.index:
        matriz.at[estado, estado] = 1.0

matriz_probabilidades = matriz.round(4)

In [0]:
canais_marketing = [c for c in matriz_probabilidades.columns if c not in ['Start', 'Null', 'Conversion']]
total_conversoes = df_modelagem['total_conversoes'].sum()

def calcular_conversao(m_prob, iteracoes=50):
    M_n = np.linalg.matrix_power(m_prob.values, iteracoes)
    idx_start = list(m_prob.index).index('Start')
    idx_conv = list(m_prob.columns).index('Conversion')
    return M_n[idx_start, idx_conv]

prob_base = calcular_conversao(matriz_probabilidades)
efeitos_remocao = {}

# Testa a remoção de cada canal e vê o impacto na conversão geral
for canal in canais_marketing:
    matriz_removida = matriz_probabilidades.copy()
    for estado in matriz_removida.index:
        prob_perdida = matriz_removida.at[estado, canal]
        matriz_removida.at[estado, canal] = 0.0
        matriz_removida.at[estado, 'Null'] += prob_perdida
        
    prob_sem_canal = calcular_conversao(matriz_removida)
    efeitos_remocao[canal] = 1 - (prob_sem_canal / prob_base) if prob_base > 0 else 0

# Distribui o crédito proporcionalmente
soma_re = sum(efeitos_remocao.values())
atribuicao = {canal: round((re / soma_re) * total_conversoes, 2) for canal, re in efeitos_remocao.items() if soma_re > 0}

df_atribuicao = pd.DataFrame(list(atribuicao.items()), columns=['canal_marketing', 'conversoes_markov'])
df_atribuicao = df_atribuicao.sort_values(by='conversoes_markov', ascending=False).reset_index(drop=True)

display(df_atribuicao)

canal_marketing,conversoes_markov
<Other,43.86
google,42.85
(direct),35.15
shop.googlemerchandisestore.com,27.91
(data deleted),20.23


In [0]:
# Salvando o resultado oficial no catálogo para plugar no Looker Studio / BI depois
spark_df_final = spark.createDataFrame(df_atribuicao)
spark_df_final.write.mode("overwrite").saveAsTable("workspace.default.resultado_markov_atribuicao")

print("Tabela gravada no Delta Lake.")

Tabela gravada no Delta Lake.


In [0]:
# Prepara a Edge List numa passada só
dados_sankey = []
for _, row in df_modelagem.iterrows():
    conversoes, caminho = int(row['total_conversoes']), row['caminho_limpo']
    if conversoes == 0 or not caminho: continue
        
    jornada = ['Start'] + caminho.split(' > ') + ['Conversion']
    for i in range(len(jornada) - 1):
        dados_sankey.append({'origem': jornada[i], 'destino': jornada[i+1], 'peso': conversoes})

df_sankey_agrupado = pd.DataFrame(dados_sankey).groupby(['origem', 'destino'])['peso'].sum().reset_index()

# Plotagem
nos_unicos = list(set(df_sankey_agrupado['origem']).union(set(df_sankey_agrupado['destino'])))
mapa_indices = {no: i for i, no in enumerate(nos_unicos)}

fig = go.Figure(data=[go.Sankey(
    node = dict(pad=20, thickness=30, line=dict(color="black", width=0.5), label=nos_unicos, color="#00B4D8"),
    link = dict(
      source=df_sankey_agrupado['origem'].map(mapa_indices).tolist(),
      target=df_sankey_agrupado['destino'].map(mapa_indices).tolist(),
      value=df_sankey_agrupado['peso'].tolist(),
      color="rgba(0, 180, 216, 0.4)" 
  ))])

fig.update_layout(title_text="Fluxo de Atribuição", font_size=14, template='plotly_dark', height=600)
fig.show()